In [77]:
from dotenv import load_dotenv
import os
import threading
import time

import requests
from anthropic import Anthropic



from pathlib import Path
import re
import base64
import json
import json5




# Load environment variables from .env file
load_dotenv()







True

In [78]:
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise Exception("ANTHROPIC_API_KEY not found in environment!")

# Verify API key format (should start with 'sk-ant-')
if not api_key.startswith('sk-ant-'):
    print("WARNING: API key doesn't start with 'sk-ant-' - this might be incorrect")

In [85]:




class AIFileParser:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)

    def _to_base64(self, file_path: Path):
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        if file_path.suffix.lower() == ".pdf":
            print("pdf content")
            with open(file_path, "rb") as f:
                return base64.b64encode(f.read()).decode("utf-8")
        else:
            print("txt content")
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read()

    def _print_processing(self, stop_event):
        while not stop_event.is_set():
            print("processing...")
            time.sleep(5)


    def json_skeleton(self, file_path):
        file_path = Path(file_path)
    
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
    
        with open("prompts/json_skeleton_prompt.txt", "r", encoding="utf-8") as f:
            prompt_text = f.read()
    
        content = self._to_base64(file_path)
    
        messages_content = []
    
        if file_path.suffix.lower() == ".pdf":
            messages_content.append({
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": content,
                }
            })
        else:
            messages_content.append({
                "type": "text",
                "text": content
            })
    
        messages_content.append({
            "type": "text",
            "text": (
                prompt_text + "\n"
                "Return only valid JSON.\n"
                "- Use only standard JSON strings with single or double quotes.\n"
                "- Do NOT use triple quotes (\"\"\" or ''' ).\n"
                "- Do NOT add extra commentary, explanations, or code fences.\n"
                "- Numbers must be numeric, not strings with symbols.\n"
                "- Use null for missing values."
            )
        })


            
        
    
        # ---- Start processing indicator ----
        stop_event = threading.Event()
        t = threading.Thread(target=self._print_processing, args=(stop_event,), daemon=True)
        t.start()
    
        try:
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=16384,
                temperature=0.2,
                messages=[
                    {
                        "role": "user",
                        "content": messages_content
                    }
                ]
            )
        finally:
            stop_event.set()
            t.join()
    
        raw_text = response.content[0].text.strip()
        # Remove backticks
        raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text, flags=re.IGNORECASE)
        raw_text = re.sub(r"\s*```$", "", raw_text)
        
        try:
            return json5.loads(raw_text)
        except Exception as e:
            raise ValueError(f"Claude returned something that could not be parsed:\n{raw_text}\nError: {e}")
        
        # Try to extract first JSON object (useful if Claude adds commentary before/after JSON)
        


    def parse_file(self, file_path, group=None):
        file_path = Path(file_path)

        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        file_ext = file_path.suffix.lower()

        if file_ext in (".pdf", ".csv", ".txt"):
            content = self._to_base64(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_ext}")

        if group is None:
            group = self._upload_group_infer(content, file_ext)
            prompt_schema = self._get_prompt_schema(group, False)
        else:
            prompt_schema = self._get_prompt_schema(group)

        return content

    def _upload_group_infer(self, content, file_ext):
        url = "http://localhost:8080/gma/v1/ai/getUploadGroups"

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        if file_ext == ".pdf":
            print("pdf group infer")

        return ""

    def _get_prompt_schema(self, group, onlyActive=True):
        url = "http://localhost:8080/gma/v1/ai/getPromptSchema"
        params = {"onlyActive": onlyActive}

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        return response.json()


In [92]:
# client = Anthropic(api_key=api_key)
# resp = client.messages.create(
#     model="claude-sonnet-4-20250514",
#     max_tokens=300,
#     temperature=0,
#     messages=[{"role": "user", "content": "say hello"}]
# )
# print(resp.content[0].text)


parser = AIFileParser(api_key = api_key)
# result = parser.parse_file("2024_dexa.pdf")
result = json.dumps(parser.json_skeleton("input_files/inbody_970.pdf"),indent=2)
print(result)

pdf content
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
{
  "upload_name": "InBody 970 Body Composition Analysis",
  "timestamp": "2021.03.31. 15:44",
  "groups": [
    {
      "group_name": "body_composition_analysis",
      "description": "Core body composition measurements including water, protein, minerals, and fat mass components.",
      "types": [
        {
          "type_name": "body_water",
          "description": "Total body water and its distribution between intracellular and extracellular compartments.",
          "names": [
            {
              "input_name": "total_body_water",
              "description": "Total body water content measurement.",
              "values": [
                {
                  "value": {
                    "name": "total_body_water",
                    "value": 27.4,
                    "unit": "L",
          

In [ ]:
'''
dexa

'''

In [88]:
'''

'''

'\n\n'

In [93]:
print(result)


{
  "upload_name": "InBody 970 Body Composition Analysis",
  "timestamp": "2021.03.31. 15:44",
  "groups": [
    {
      "group_name": "body_composition_analysis",
      "description": "Core body composition measurements including water, protein, minerals, and fat mass components.",
      "types": [
        {
          "type_name": "body_water",
          "description": "Total body water and its distribution between intracellular and extracellular compartments.",
          "names": [
            {
              "input_name": "total_body_water",
              "description": "Total body water content measurement.",
              "values": [
                {
                  "value": {
                    "name": "total_body_water",
                    "value": 27.4,
                    "unit": "L",
                    "normalized_value": 27.4,
                    "normalized_unit": "L",
                    "measurement_type": "quantity",
                    "date": "2021.03.31",
      